In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import imageio
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import optimizers

In [ ]:
df = pd.read_csv('full_df_cleaned_v3.csv')

In [ ]:
df.head()

In [ ]:
df_n = df[df['tarstr']=='N']

In [ ]:
df_n.shape

In [ ]:
df_c = df[df['tarstr']=='C']

In [ ]:
df_c.shape

In [ ]:
df_N_C = pd.concat([df_n, df_c], ignore_index=True)

In [ ]:
df_N_C.shape

In [ ]:
IMAGE_PATH = '../projectmini/preprocessed_images2/'

In [ ]:
df_N_C['filepath'] = IMAGE_PATH + df_N_C['filename']

In [ ]:
import imageio
img_data = []
number_id_nofile = []

for i in range(len(df_N_C)):
  try:
    img_data.append(imageio.imread(df_N_C['filepath'][i]))
  except:
    number_id_nofile.append(df_N_C.index[i])

In [ ]:
df_N_C = df_N_C.drop(number_id_nofile)

In [ ]:
df_N_C['tarstr'].value_counts()

In [ ]:
img_data_array = np.array(img_data)

In [ ]:
img_data_array.shape

In [ ]:
X = img_data_array

In [ ]:
y = df_N_C['C']

In [ ]:
y.shape

In [ ]:
y.value_counts(1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y)

In [ ]:
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
df_N_C['C'].value_counts()

In [ ]:
y_train.value_counts(1)

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16

def load_model():
   model = VGG16(include_top=False, weights='imagenet',
                input_shape=(256, 256, 3),
   model.add(layers.Conv2D(32, (3,3), input_shape=(256, 256, 3), activation='relu', padding='same')),
   model.add(layers.MaxPool2D(pool_size=(2,2))),
   model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same')),
   model.add(layers.MaxPool2D(pool_size=(2,2))),
   model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same')),
   model.add(layers.MaxPool2D(pool_size=(3,3))),
   model.add(layers.Conv2D(256, (3,3), activation='relu', padding='same')),
   model.add(layers.MaxPool2D(pool_size=(3,3))),
   ### Flattening
   model.add(layers.Flatten()),
   ### One fully connected
   model.add(layers.Dense(120, activation='relu')),
   model.add(layers.Dropout(rate=0.5)),
   model.add(layers.Dense(60, activation='relu')),
   model.add(layers.Dropout(rate=0.5)),
   model.add(layers.Dense(1, activation='sigmoid'))

                )
   adam_opt = optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999)

   model.compile(loss='binary_crossentropy', 
         optimizer=adam_opt,
         metrics=['accuracy'])

   return model

In [ ]:
 model = initialize_model()
 es = EarlyStopping(patience=20, restore_best_weights=True)

 history = model.fit(X_train, y_train,
                     validation_split=0.3,
                     epochs=20,
                     batch_size=16, 
                     verbose=1,
                     callbacks=[es])

In [ ]:
 print(model.evaluate(X_test, y_test, verbose=0))

In [ ]:
 def plot_history(history, title='', axs=None, exp_name=""):
     if axs is not None:
         ax1, ax2 = axs
     else:
         f, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
     if len(exp_name) > 0 and exp_name[0] != '_':
         exp_name = '_' + exp_name
     ax1.plot(history.history['loss'], label='train' + exp_name)
     ax1.plot(history.history['val_loss'], label='val' + exp_name)
     ax1.set_ylim(0., 2.2)
     ax1.set_title('loss')
     ax1.legend()

     ax2.plot(history.history['accuracy'], label='train accuracy'  + exp_name)
     ax2.plot(history.history['val_accuracy'], label='val accuracy'  + exp_name)
     ax2.set_ylim(0.25, 1.)
     ax2.set_title('Accuracy')
     ax2.legend()
     return (ax1, ax2)

# # YOUR CODE HERE
 plot_history(history, title='', axs=None, exp_name="")

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16

def load_model():
    
 
  model = VGG16(include_top=False, weights='imagenet',
                input_shape=(256, 256, 3)
                  )
    
  return model

model = load_model()

In [ ]:
def set_nontrainable_layers(model):
    # YOUR CODE HERE
  model.trainable=False
  return model

In [ ]:
def add_last_layers(model):
  # YOUR CODE HERE
  base_model = set_nontrainable_layers(model)
  flattening_layer = layers.Flatten()
  dense_layer = layers.Dense(500, activation='relu')
  prediction_layer = layers.Dense(1, activation='sigmoid')
    
  model = models.Sequential([
    base_model,
    flattening_layer,
    dense_layer,
    prediction_layer
  ])

  return model

In [ ]:
model = load_model()
model = add_last_layers(model)

In [ ]:
def compile_model(model):
  adam_opt = optimizers.Adam(learning_rate=0.0001)

  model.compile(loss='binary_crossentropy',
              optimizer=adam_opt,
              metrics=['accuracy'])
  return model

model = compile_model(model)

In [ ]:
def build_model():
    # YOUR CODE HERE
    model = load_model()
    model = add_last_layers(model)
    model = compile_model(model)

    return model

model = build_model()

In [ ]:
model.summary()

In [ ]:
es = EarlyStopping(patience=20,  restore_best_weights=True)

model = build_model()

history = model.fit(X_train, y_train,
                    validation_split=0.3,
                    epochs=20,
                    batch_size=16, 
                    verbose=1, 
                    callbacks=[es])